In [ ]:
import json
import re
import shutil
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import kaggle_benchmarks as kbench

TASK_JSON_RELATIVE_PATH = "benchmark/cuc_metacognition_v4_candidate.task.json"
PARAMS_JSON_RELATIVE_PATH = "benchmark/cuc_v4_candidate_params.json"
INPUT_ROOTS = [Path("/kaggle/input"), Path("/mnt/data")]
EXPORT_ROOT = Path("/kaggle/working/cuc_v4_candidate_export")
CASE_LIMIT = None

SYSTEM_PROMPT = """You are a deterministic evidence-revision analyst. Evidence packs may be rendered as legal memoranda, incident digests, board updates, operations briefs, procurement memos, or compliance memoranda. Treat those surface differences as formatting only. Use only claims and artifact IDs present in the packs. Revise only what changed. Preserve stable sibling branches unless new contrary evidence is introduced. Track newly created unknowns explicitly. Never invent evidence citations."""

RUN1_TEMPLATE = """You are evaluating the CLEAN CUC v4 candidate evidence pack below.

Return a concise analyst snapshot. It does not have to be valid JSON, but keep it structured and compact.

Include:
- ACTIVE_CLAIMS: claims that currently look most decision-relevant
- REVISION_TARGETS: claims most likely to need revision if a key artifact changes
- PRESERVE_WATCH: stable sibling claims that should not be rewritten without new contrary evidence
- OPEN_UNKNOWNS: unresolved issues that would materially change the downstream assessment

=== CLEAN PACK ===
{clean_pack_text}
=== END CLEAN PACK ===
"""

RUN2_TEMPLATE = """The evidence pack has been perturbed after your first analysis.

Use your prior analysis plus the perturbed pack to output STRICT JSON only.
Do not wrap it in markdown. Do not add commentary.
Use only IDs, claim labels, and evidence artifacts present in the perturbed pack.

Required JSON schema:
{
  "q1_delta_detection": {
    "summary": "string",
    "changed_state_ids": ["state:..."],
    "changed_event_ids": ["event:..."],
    "changed_edge_ids": ["edge:..."],
    "changed_scenario_ids": ["scenario:..."]
  },
  "q2_revision": {
    "preserve_ids": ["id"],
    "weaken_ids": ["id"],
    "strengthen_ids": ["id"],
    "retract_ids": ["id"],
    "newly_relevant_unknown_ids": ["unknown:..."],
    "forbidden_revision_ack": ["id"]
  },
  "q3_unknown_ledger": {
    "resolved_unknown_ids": ["unknown:..."],
    "still_open_unknown_ids": ["unknown:..."],
    "new_unknown_ids": ["unknown:..."],
    "category_map": {"unknown:id": "resolved|still_open|new"},
    "summary": "string"
  },
  "citations": {
    "id": ["artifact_or_evidence_id"],
    "another_id": ["artifact_or_evidence_id"]
  },
  "self_check": {
    "no_global_rewrite_claimed": true,
    "max_hop_claimed": 0,
    "low_confidence_ids": ["id"]
  }
}

=== PERTURBED PACK ===
{perturbed_pack_text}
=== END PERTURBED PACK ===
"""


def resolve_artifact_path(relative_path: str) -> Path:
    local_candidate = Path(relative_path)
    if local_candidate.is_file():
        return local_candidate

    for input_root in INPUT_ROOTS:
        if not input_root.exists():
            continue
        exact_candidate = input_root / relative_path
        if exact_candidate.is_file():
            return exact_candidate
        matches = sorted(
            input_root.rglob(Path(relative_path).name),
            key=lambda p: (len(str(p)), str(p)),
        )
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not resolve artifact path for {relative_path}")


TASK_JSON_PATH = resolve_artifact_path(TASK_JSON_RELATIVE_PATH)
PARAMS_JSON_PATH = resolve_artifact_path(PARAMS_JSON_RELATIVE_PATH)

print("Resolved TASK_JSON_PATH:", TASK_JSON_PATH)
print("Resolved PARAMS_JSON_PATH:", PARAMS_JSON_PATH)


In [ ]:
REQUIRED_TOP_KEYS = {"q1_delta_detection", "q2_revision", "q3_unknown_ledger", "citations", "self_check"}
REQUIRED_Q1_KEYS = {"summary", "changed_state_ids", "changed_event_ids", "changed_edge_ids", "changed_scenario_ids"}
REQUIRED_Q2_KEYS = {"preserve_ids", "weaken_ids", "strengthen_ids", "retract_ids", "newly_relevant_unknown_ids", "forbidden_revision_ack"}
REQUIRED_Q3_KEYS = {"resolved_unknown_ids", "still_open_unknown_ids", "new_unknown_ids", "category_map", "summary"}
REQUIRED_SELF_CHECK_KEYS = {"no_global_rewrite_claimed", "max_hop_claimed", "low_confidence_ids"}


def extract_json_object(text: str):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, end = decoder.raw_decode(text[i:])
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    raise ValueError("No JSON object found in response")


def is_list_of_str(value):
    return isinstance(value, list) and all(isinstance(x, str) for x in value)


def validate_response_shape(data):
    errors = []
    if not isinstance(data, dict):
        return ["top_level_not_object"]
    if set(data.keys()) != REQUIRED_TOP_KEYS:
        errors.append("top_level_keys_mismatch")
    q1 = data.get("q1_delta_detection", {})
    q2 = data.get("q2_revision", {})
    q3 = data.get("q3_unknown_ledger", {})
    sc = data.get("self_check", {})
    citations = data.get("citations", {})
    if set(q1.keys()) != REQUIRED_Q1_KEYS:
        errors.append("q1_keys_mismatch")
    if set(q2.keys()) != REQUIRED_Q2_KEYS:
        errors.append("q2_keys_mismatch")
    if set(q3.keys()) != REQUIRED_Q3_KEYS:
        errors.append("q3_keys_mismatch")
    if set(sc.keys()) != REQUIRED_SELF_CHECK_KEYS:
        errors.append("self_check_keys_mismatch")
    for field in ["changed_state_ids", "changed_event_ids", "changed_edge_ids", "changed_scenario_ids"]:
        if field in q1 and not is_list_of_str(q1[field]):
            errors.append(f"q1.{field}_not_list_of_str")
    for field in ["preserve_ids", "weaken_ids", "strengthen_ids", "retract_ids", "newly_relevant_unknown_ids", "forbidden_revision_ack"]:
        if field in q2 and not is_list_of_str(q2[field]):
            errors.append(f"q2.{field}_not_list_of_str")
    for field in ["resolved_unknown_ids", "still_open_unknown_ids", "new_unknown_ids"]:
        if field in q3 and not is_list_of_str(q3[field]):
            errors.append(f"q3.{field}_not_list_of_str")
    if "category_map" in q3 and not isinstance(q3["category_map"], dict):
        errors.append("q3.category_map_not_object")
    if "summary" in q1 and not isinstance(q1["summary"], str):
        errors.append("q1.summary_not_string")
    if "summary" in q3 and not isinstance(q3["summary"], str):
        errors.append("q3.summary_not_string")
    if not isinstance(citations, dict):
        errors.append("citations_not_object")
    else:
        for k, v in citations.items():
            if not isinstance(k, str) or not is_list_of_str(v):
                errors.append("citations_invalid_entry")
                break
    if "no_global_rewrite_claimed" in sc and not isinstance(sc["no_global_rewrite_claimed"], bool):
        errors.append("self_check.no_global_rewrite_claimed_not_bool")
    if "max_hop_claimed" in sc and not isinstance(sc["max_hop_claimed"], int):
        errors.append("self_check.max_hop_claimed_not_int")
    if "low_confidence_ids" in sc and not is_list_of_str(sc["low_confidence_ids"]):
        errors.append("self_check.low_confidence_ids_not_list_of_str")
    return errors


def collect_valid_ids(*packs):
    valid_ids = set()
    evidence_ids = set()
    for pack in packs:
        sections = pack.get("sections", {})
        for section_name, items in sections.items():
            if not isinstance(items, list):
                continue
            for item in items:
                if not isinstance(item, dict):
                    continue
                for key, value in item.items():
                    if key.endswith("_id") and isinstance(value, str):
                        valid_ids.add(value)
                if section_name == "evidence" and isinstance(item.get("evidence_id"), str):
                    evidence_ids.add(item["evidence_id"])
    return valid_ids, evidence_ids


def expected_values(expected_delta, field_name):
    mapping = {
        "changed_state_ids": {item["id"] for item in expected_delta.get("changed_states", [])},
        "changed_event_ids": {item["id"] for item in expected_delta.get("changed_events", [])},
        "changed_edge_ids": {item["id"] for item in expected_delta.get("changed_edges", [])},
        "changed_scenario_ids": {item["scenario_id"] for item in expected_delta.get("scenario_rank_changes", [])},
        "newly_relevant_unknown_ids": {item["id"] for item in expected_delta.get("new_unknowns", [])},
        "new_unknown_ids": {item["id"] for item in expected_delta.get("new_unknowns", [])},
        "resolved_unknown_ids": {item["id"] for item in expected_delta.get("resolved_unknowns", [])},
        "forbidden_revision_ack": set(expected_delta.get("forbidden_revisions", [])),
    }
    return mapping.get(field_name, set())


def set_f1(pred, gold):
    pred = set(pred)
    gold = set(gold)
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    tp = len(pred & gold)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold) if gold else 0.0
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def set_accuracy(pred, gold):
    pred = set(pred)
    gold = set(gold)
    if not pred and not gold:
        return 1.0
    if not pred:
        return 0.0
    return len(pred & gold) / len(pred)


def citation_coverage(response, expected_delta, valid_evidence_ids):
    citations = response.get("citations", {})
    support_map = expected_delta.get("supporting_evidence_map", {})
    target_ids = set()
    target_ids |= {item["id"] for item in expected_delta.get("changed_states", [])}
    target_ids |= {item["id"] for item in expected_delta.get("changed_events", [])}
    target_ids |= {item["id"] for item in expected_delta.get("changed_edges", [])}
    target_ids |= {item["id"] for item in expected_delta.get("new_unknowns", [])}
    target_ids |= {item["id"] for item in expected_delta.get("resolved_unknowns", [])}
    if not target_ids:
        return 1.0
    covered = 0
    for item_id in target_ids:
        cited = citations.get(item_id, [])
        if not isinstance(cited, list):
            continue
        cited = [x for x in cited if isinstance(x, str) and x in valid_evidence_ids]
        if not cited:
            continue
        expected_evidence = set(support_map.get(item_id, []))
        if not expected_evidence or expected_evidence.intersection(cited):
            covered += 1
    return covered / len(target_ids)


def score_structured_response(response_text, expected_delta_json, scoring_manifest_json, clean_pack_json, perturbed_pack_json):
    hard_fail_reasons = []
    parse_ok = True
    try:
        response = extract_json_object(response_text)
    except Exception:
        parse_ok = False
        response = {}
        hard_fail_reasons.append("invalid_json")

    schema_errors = [] if parse_ok else ["invalid_json"]
    if parse_ok:
        schema_errors = validate_response_shape(response)
        if schema_errors:
            hard_fail_reasons.append("response_schema_violation")

    expected_delta = json.loads(expected_delta_json) if isinstance(expected_delta_json, str) else expected_delta_json
    scoring_manifest = json.loads(scoring_manifest_json) if isinstance(scoring_manifest_json, str) else scoring_manifest_json
    clean_pack = json.loads(clean_pack_json) if isinstance(clean_pack_json, str) else clean_pack_json
    perturbed_pack = json.loads(perturbed_pack_json) if isinstance(perturbed_pack_json, str) else perturbed_pack_json
    valid_ids, valid_evidence_ids = collect_valid_ids(clean_pack, perturbed_pack)

    predicted_ids = set()
    if isinstance(response, dict):
        for bucket in [response.get("q1_delta_detection", {}), response.get("q2_revision", {}), response.get("q3_unknown_ledger", {})]:
            if not isinstance(bucket, dict):
                continue
            for value in bucket.values():
                if isinstance(value, list):
                    predicted_ids.update(x for x in value if isinstance(x, str))
                elif isinstance(value, dict):
                    predicted_ids.update(k for k in value.keys() if isinstance(k, str))
    if any(item_id not in valid_ids for item_id in predicted_ids):
        hard_fail_reasons.append("fabricated_ids")

    citations = response.get("citations", {}) if isinstance(response, dict) else {}
    cited_evidence = {ev for vals in citations.values() if isinstance(vals, list) for ev in vals if isinstance(ev, str)}
    if any(ev not in valid_evidence_ids for ev in cited_evidence):
        hard_fail_reasons.append("fabricated_evidence_ids")

    q2 = response.get("q2_revision", {}) if isinstance(response, dict) else {}
    preserve_ids = set(q2.get("preserve_ids", [])) if isinstance(q2, dict) else set()
    retract_ids = set(q2.get("retract_ids", [])) if isinstance(q2, dict) else set()
    if preserve_ids & retract_ids:
        hard_fail_reasons.append("preserve_and_retract_same_id")

    q1 = response.get("q1_delta_detection", {}) if isinstance(response, dict) else {}
    q3 = response.get("q3_unknown_ledger", {}) if isinstance(response, dict) else {}

    metric_scores = {
        "changed_states": 1.0 if set(q1.get("changed_state_ids", [])) == expected_values(expected_delta, "changed_state_ids") else 0.0,
        "changed_events": 1.0 if set(q1.get("changed_event_ids", [])) == expected_values(expected_delta, "changed_event_ids") else 0.0,
        "changed_edges": set_f1(q1.get("changed_edge_ids", []), expected_values(expected_delta, "changed_edge_ids")),
        "changed_scenarios": 1.0 if set(q1.get("changed_scenario_ids", [])) == expected_values(expected_delta, "changed_scenario_ids") else 0.0,
        "new_unknowns_q2": set_f1(q2.get("newly_relevant_unknown_ids", []), expected_values(expected_delta, "newly_relevant_unknown_ids")),
        "new_unknowns_q3": set_f1(q3.get("new_unknown_ids", []), expected_values(expected_delta, "new_unknown_ids")),
        "resolved_unknowns_q3": set_f1(q3.get("resolved_unknown_ids", []), expected_values(expected_delta, "resolved_unknown_ids")),
        "forbidden_revision_ack": set_accuracy(q2.get("forbidden_revision_ack", []), expected_values(expected_delta, "forbidden_revision_ack")),
    }

    axis_scores = {
        "delta_detection": round((metric_scores["changed_states"] + metric_scores["changed_events"]) / 2, 6),
        "causal_temporal_propagation": round((metric_scores["changed_edges"] + metric_scores["changed_scenarios"]) / 2, 6),
        "selective_revision": round((metric_scores["new_unknowns_q2"] + metric_scores["forbidden_revision_ack"]) / 2, 6),
        "unknown_ledger_discipline": round((metric_scores["new_unknowns_q3"] + metric_scores["resolved_unknowns_q3"]) / 2, 6),
        "evidence_grounding": round(citation_coverage(response, expected_delta, valid_evidence_ids), 6),
    }

    penalty_collateral = 0.0
    forbidden = set(expected_delta.get("forbidden_revisions", []))
    if retract_ids:
        penalty_collateral = len(retract_ids & forbidden) / max(len(retract_ids), 1)
    gold_new_unknowns = expected_values(expected_delta, "new_unknown_ids")
    pred_new_unknowns = set(q3.get("new_unknown_ids", [])) if isinstance(q3, dict) else set()
    penalty_missed_second_order = (len(gold_new_unknowns - pred_new_unknowns) / len(gold_new_unknowns)) if gold_new_unknowns else 0.0

    weights = scoring_manifest.get("weights", {})
    weighted_total = 0.0
    total_weight = 0.0
    for axis, score in axis_scores.items():
        weight = float(weights.get(axis, 0.0))
        weighted_total += score * weight
        total_weight += weight
    overall = weighted_total / total_weight if total_weight else 0.0
    overall = max(0.0, overall - 0.75 * penalty_collateral - 0.75 * penalty_missed_second_order)

    pass_rule = scoring_manifest.get("operational_pass_rule", {})
    overall_pass = (
        not hard_fail_reasons and
        overall >= float(pass_rule.get("overall_min", 0.72)) and
        axis_scores["delta_detection"] >= float(pass_rule.get("delta_detection_min", 0.5)) and
        axis_scores["evidence_grounding"] >= float(pass_rule.get("citation_coverage_min", 0.5))
    )

    return {
        "parse_ok": parse_ok,
        "schema_ok": not schema_errors,
        "schema_errors": schema_errors,
        "hard_fail_reasons": sorted(set(hard_fail_reasons)),
        "metric_scores": {k: round(v, 6) for k, v in metric_scores.items()},
        "axis_scores": axis_scores,
        "overall_score": round(overall, 6),
        "overall_pass": overall_pass,
    }


In [ ]:
task_spec = json.loads(TASK_JSON_PATH.read_text(encoding="utf-8"))
TASK_NAME = task_spec["name"]
assert TASK_NAME == "cuc_metacognition_v4_candidate"

exec(task_spec["definition"], globals())
assert "cuc_metacognition_v4_candidate" in globals()

print("Loaded task:", TASK_NAME)
print("Task definition path:", TASK_JSON_PATH)


In [ ]:
records = json.loads(PARAMS_JSON_PATH.read_text(encoding="utf-8"))
assert isinstance(records, list) and records, "Params JSON must contain at least one record"

if CASE_LIMIT is not None:
    records = records[:CASE_LIMIT]

df = pd.DataFrame(records)
assert not df.empty, "No v4 candidate rows loaded"
required_columns = {
    "case_id",
    "family_id",
    "render_profile",
    "hop_depth",
    "clean_pack_text",
    "perturbed_pack_text",
    "clean_pack_json",
    "perturbed_pack_json",
    "expected_delta_json",
    "causal_chain_json",
    "scoring_manifest_json",
}
missing = sorted(required_columns.difference(df.columns))
assert not missing, f"Missing required columns: {missing}"

print("Loaded rows:", len(df))
print(df[["case_id", "family_id", "sector_skin", "render_profile", "hop_depth"]].to_string(index=False))


In [ ]:
assert not df.empty, "df must not be empty"
sample_row = df.iloc[0]

rendered_run1 = RUN1_TEMPLATE.replace("{clean_pack_text}", sample_row["clean_pack_text"])
rendered_run2 = RUN2_TEMPLATE.replace("{perturbed_pack_text}", sample_row["perturbed_pack_text"])
assert "{clean_pack_text}" not in rendered_run1
assert "{perturbed_pack_text}" not in rendered_run2

sample_score = score_structured_response(
    json.dumps({
        "q1_delta_detection": {
            "summary": "smoke",
            "changed_state_ids": [],
            "changed_event_ids": [],
            "changed_edge_ids": [],
            "changed_scenario_ids": []
        },
        "q2_revision": {
            "preserve_ids": [],
            "weaken_ids": [],
            "strengthen_ids": [],
            "retract_ids": [],
            "newly_relevant_unknown_ids": [],
            "forbidden_revision_ack": []
        },
        "q3_unknown_ledger": {
            "resolved_unknown_ids": [],
            "still_open_unknown_ids": [],
            "new_unknown_ids": [],
            "category_map": {},
            "summary": "smoke"
        },
        "citations": {},
        "self_check": {
            "no_global_rewrite_claimed": True,
            "max_hop_claimed": 0,
            "low_confidence_ids": []
        }
    }),
    expected_delta_json=sample_row["expected_delta_json"],
    scoring_manifest_json=sample_row["scoring_manifest_json"],
    clean_pack_json=sample_row["clean_pack_json"],
    perturbed_pack_json=sample_row["perturbed_pack_json"],
)
assert "overall_score" in sample_score and "axis_scores" in sample_score
print("Smoke test passed for sample case:", sample_row["case_id"])

results = cuc_metacognition_v4_candidate.evaluate(
    llm=[kbench.llm],
    evaluation_data=df,
)

print(results)
for run in results.runs:
    print(run)


In [ ]:
def iso_or_none(value):
    if value is None:
        return None
    return value.isoformat() if hasattr(value, "isoformat") else str(value)


def safe_status(value):
    if value is None:
        return None
    return getattr(value, "value", str(value))


def summarize_assertion(assertion):
    return {
        "id": getattr(assertion, "id", None),
        "passed": bool(getattr(assertion, "passed", False)),
        "expectation": getattr(assertion, "expectation", None),
    }


def summarize_run(run):
    params = getattr(run, "params", {}) or {}
    llm_obj = params.get("llm")
    result = bool(getattr(run, "result", getattr(run, "passed", False)))
    return {
        "id": getattr(run, "id", None),
        "param_id": getattr(run, "param_id", None),
        "case_id": params.get("case_id", getattr(run, "case_id", None)),
        "family_id": params.get("family_id"),
        "sector_skin": params.get("sector_skin"),
        "render_profile": params.get("render_profile"),
        "hop_depth": params.get("hop_depth"),
        "named_evidence_artifact": params.get("named_evidence_artifact"),
        "llm": getattr(llm_obj, "name", str(llm_obj)) if llm_obj is not None else getattr(run, "llm_name", None),
        "result": result,
        "passed": result,
        "status": safe_status(getattr(run, "status", None)),
        "error_message": getattr(run, "error_message", None),
        "start_time": iso_or_none(getattr(run, "start_time", None)),
        "end_time": iso_or_none(getattr(run, "end_time", None)),
        "assertion_results": [
            summarize_assertion(a) for a in getattr(run, "assertion_results", [])
        ],
    }


EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(TASK_JSON_PATH, EXPORT_ROOT / TASK_JSON_PATH.name)
shutil.copy2(PARAMS_JSON_PATH, EXPORT_ROOT / PARAMS_JSON_PATH.name)

preview_columns = [
    "param_id",
    "case_id",
    "family_id",
    "sector_skin",
    "render_profile",
    "hop_depth",
    "named_evidence_artifact",
]
preview_df = df[preview_columns].copy()
preview_df.to_csv(EXPORT_ROOT / "evaluation_input_preview.csv", index=False)

runs_summary = [summarize_run(run) for run in getattr(results, "runs", [])]

export_payload = {
    "task_name": TASK_NAME,
    "task_json_relative_path": TASK_JSON_RELATIVE_PATH,
    "params_json_relative_path": PARAMS_JSON_RELATIVE_PATH,
    "resolved_task_json_path": str(TASK_JSON_PATH),
    "resolved_params_json_path": str(PARAMS_JSON_PATH),
    "exported_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "case_count": len(df),
    "run_count": len(runs_summary),
    "pass_count": sum(1 for run in runs_summary if run["result"]),
    "fail_count": sum(1 for run in runs_summary if not run["result"]),
    "results_repr": repr(results),
    "runs": runs_summary,
}

(EXPORT_ROOT / "results_export.json").write_text(
    json.dumps(export_payload, indent=2, sort_keys=True),
    encoding="utf-8",
)
(EXPORT_ROOT / "results_repr.txt").write_text(f"{results}\n", encoding="utf-8")

print("Exported notebook stub artifacts to:", EXPORT_ROOT)
for path in sorted(EXPORT_ROOT.iterdir()):
    print(" -", path)
